## Example 1: Date parsing and time formatting

In [ ]:
from jupyterhub_idle_culler import parse_date, format_td, utcnow
from datetime import datetime, timedelta, timezone

# Example 1: Date parsing and time formatting for idle detection
print("Example 1: Date parsing and time formatting")

# Get current time
current_time = utcnow()
print(f"Current UTC time: {current_time}")

# Parse different date formats
date_strings = [
    "2023-12-01T10:30:00Z",
    "2023-12-01 10:30:00",
    "2023-12-01T10:30:00.123456Z"
]

expected_instants = {
    "2023-12-01T10:30:00Z": datetime(2023, 12, 1, 10, 30, tzinfo=timezone.utc),
    "2023-12-01 10:30:00": datetime(2023, 12, 1, 10, 30, tzinfo=timezone.utc),
    "2023-12-01T10:30:00.123456Z": datetime(2023, 12, 1, 10, 30, 0, 123456, tzinfo=timezone.utc),
}

for date_str in date_strings:
    parsed = parse_date(date_str)
    print(f"Parsed '{date_str}': {parsed}")
    assert parsed == expected_instants[date_str], (date_str, parsed)

# Format time deltas
deltas = [timedelta(minutes=30), timedelta(hours=2), timedelta(days=1)]
for td in deltas:
    formatted = format_td(td)
    print(f"Formatted {td}: {formatted}")
    assert formatted == {timedelta(minutes=30): "00:30:00", timedelta(hours=2): "02:00:00", timedelta(days=1): "24:00:00"}[td], formatted

print("Example 1 completed successfully")

## Example 2: Configuration and options setup

In [ ]:
# Example 2: Configuration via the IdleCuller traitlets Application.
# v2.0 replaced the module-level tornado.options define/options/parse_command_line
# pattern with a traitlets Application — config is traits on the IdleCuller class.
from jupyterhub_idle_culler import IdleCuller

culler = IdleCuller()
print(f"IdleCuller application: {type(culler).__name__}")

# Defaults
print(f"Default timeout: {culler.timeout} seconds")
print(f"Default cull_every: {culler.cull_every} seconds")
print(f"Default max_age: {culler.max_age} seconds")

# Configure the same knobs the CLI exposes (--timeout / --cull-every / --max-age)
culler.timeout = 900
culler.cull_every = 120
culler.max_age = 7200

print(f"Configured timeout: {culler.timeout} seconds")
print(f"Configured cull interval: {culler.cull_every} seconds")
print(f"Configured max age: {culler.max_age} seconds")
assert (culler.timeout, culler.cull_every, culler.max_age) == (900, 120, 7200)

print("Example 2 completed successfully")

## Example 3: HTTP client and SSL context setup

In [ ]:
from jupyterhub_idle_culler import make_ssl_context, AsyncHTTPClient, HTTPRequest
import ssl

# Example 3: HTTP client and SSL context setup
print("Example 3: HTTP client and SSL context setup")

# Create SSL context for secure connections
ssl_context = make_ssl_context(
    keyfile=None,
    certfile=None, 
    cafile=None
)

if ssl_context:
    print(f"SSL context created: {type(ssl_context).__name__}")
    print(f"SSL protocol: {ssl_context.protocol}")
else:
    print("SSL context: None (default SSL settings)")
    ssl_context = ssl.create_default_context()
    print(f"Default SSL context created: {type(ssl_context).__name__}")

# Setup HTTP client for JupyterHub API calls
http_client = AsyncHTTPClient()
print(f"HTTP client created: {type(http_client).__name__}")

# Create sample HTTP request for JupyterHub API
api_url = "https://example-hub.com/hub/api/users"
request = HTTPRequest(
    url=api_url,
    method="GET",
    headers={"Authorization": "Bearer fake-token"}
)

print(f"HTTP request configured:")
print(f"  URL: {request.url}")
print(f"  Method: {request.method}")
print(f"  Headers configured: {bool(request.headers)}")

print("Example 3 completed successfully")

## Example 4: URL construction and request handling

In [ ]:
from jupyterhub_idle_culler import url_concat, quote
from urllib.parse import urlencode

# Example 4: URL construction and request handling
print("Example 4: URL construction and request handling")

# Base JupyterHub API URL
base_url = "https://hub.example.com/hub/api"

# Build URLs for different API endpoints
endpoints = {
    "users": "/users",
    "servers": "/users/{user}/servers", 
    "proxy": "/proxy"
}

# Construct URLs with parameters
params = {"state": "running", "include_stopped_servers": "1"}
users_url = url_concat(base_url + endpoints["users"], params)
print(f"Users API URL: {users_url}")

# Quote special characters in usernames
usernames = ["user@domain.com", "user with spaces", "user/special"]
for username in usernames:
    quoted = quote(username, safe='')
    server_url = base_url + endpoints["servers"].format(user=quoted)
    print(f"Server URL for '{username}': {server_url}")

# Build query string manually
query_params = {"timeout": 600, "cull_every": 60}
query_string = urlencode(query_params)
full_url = f"{base_url}/config?{query_string}"
print(f"Config URL: {full_url}")

print("Example 4 completed successfully")

## Example 5: Periodic callback and event loop setup

In [ ]:
# Example 5: Periodic callback and event loop setup.
# v2.0 no longer re-exports tornado's app_log — use the IdleCuller app's own
# traitlets logger instead.
from datetime import timedelta

from jupyterhub_idle_culler import IdleCuller, IOLoop, PeriodicCallback

culler = IdleCuller()
log = culler.log
log.setLevel(10)  # DEBUG level
print(f"Logger configured: {log.name}")

# Define callback function for periodic culling
def cull_servers():
    """Simulate the server culling process"""
    log.info("Checking for idle servers...")
    log.info("Found 3 idle servers")
    log.info("Culled 2 servers successfully")

# Create periodic callback
callback_interval = timedelta(minutes=5).total_seconds() * 1000  # milliseconds
periodic_culler = PeriodicCallback(cull_servers, callback_interval)
print(f"Periodic callback created with {callback_interval/1000}s interval")

# Get IOLoop instance
io_loop = IOLoop.current()
print(f"IOLoop instance: {type(io_loop).__name__}")

# Demonstrate callback setup (without starting)
print(f"Callback ready: {not periodic_culler.is_running()}")
print(f"Callback interval: {periodic_culler.callback_time}ms")

# Simulate one execution
cull_servers()

print("Example 5 completed successfully")